# Active Recall

How do I best remember the key aspects of an implementation so that I can reproduce a piece of code from scratch?<br/>
Right now my thoughts are
- Try to remember the tests name and make sure the test name really connects with what were are trying to do. Here, we test ourselves by stubbing
  out all the test by memory writing in pseudo the steps involved in the test implementation. Ensure we relate everything back to the overall goal.
  A useful tactic here is having like guiding questions that help us tie everything together. What, Why, How, Where, When questions.
- Taking at page from how artists generally produce or reproduce a piece of art quickly
    1. Stub out a general outline of a function/method, struct, `impl` block or trait 
    2. Add initial general detail
    3. Add finer details last

Although it might look like we are trying to memorize the code word for word, they key idea here is how best to fill in the blanks when working on an implementation.<br/>
i.e. what are they key details that we all blanking out on, why, and why are the details important?

My assumption is that this will help in structuring our thinking more clearly and make our own implementation straightforwad because we have a framework in place that
helps us layout our implementations, from high level to low level details.

Honestly, the more I think of this the more it turns out that the leetcode style of solving problems is what we are using to think about project implementation.

# To Revisit

There are couple of topics that I feel I have to revesit to consider this book complete
- Proper CI/CD
- Handling edge cases raised in Chapter 7
- Proper documentation of the deployment to fly.io
- Adequate error handling logging and tracing that we got a taste of in chapter 8.

# 7.0 Reject Invalid Subscriber #2

## 7.7. Sending A Confirmation Email

### 7.7.x Actix

#### 7.7.1. Static Email

##### 7.7.1.0. Oveview

_**Why?**_<br/>
Check that when we get a new subscriber they get an confirmation email.

_**How?**_<br/>
- First add a test that checks a new subscriber is sent an email. We'll use `wiremock::Mock` to mock the email server that mocks sending
  and email and returning a `200 OK` if it receives the send email request.
- Extract the email client into `subscribe` in order to call `email_client.send_emai()`

##### 7.7.1.1. Red Test

_**What?**_
- What is the name of the test?
  - expected - `subscribe_send_email_for_valid_form_data`
  - actual - `subscribe_sends_confirmation_email_for_valid_form_data`
  - diff - _`sends_confirmation`_
- Stub out solution from memory
```Rust
//! tests/api/helpers.rs ✅

use wiremock::MockServer;

use zero2prod::email_client::EmailClient;

pub struct TestApp {
    pub address: String,
    pub db_pool: PgPool,
    pub email_server: MockServer
}

pub async fn spawn_app() -> TestApp {
    // [...]
    let email_server = MockServer::start().await;
    let configuration = {
        let config = get_config().expect("Failed to read configuration files");
        // [...]
        config.email_client.base_url = &email_server.uri();
        config
    };
    TestApp {
        // [...]
        email_server
    }
}
```
```Rust
//! test/api/subscriptions.rs

#[tokio::test]
async fn subscribe_sends_confirmation_email_on_valid_form_data() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .expect(1) // -> Assertion will be done at tne end of the scope
        .mount(&app.email_server)
        .await; ⚠️

    // Act
    app.post_subscriptions(&body.into()).await; ⚠️

    // Assert
    // Mocke assert on drop ⚠️
}
```
- Update markdown with screenshot.

##### 7.7.1.2. Green Test

_**What?**_<br/>
- Capture `email_client` in `subscribe`
- Make `email_client.send_email` call with dummy data

**Expected Implementation**
```Rust
//! src/routes/subscriptions.rs

pub async fn subscribe(
    form: Form<FormData>, // ?? ❌
    db_pool: web::Data<PgPool>,
    email_client: web::Data<EmailClient>,
) -> HttpResponse {

    // [...]
    
    let new_subscriber = match FormData { // ?? ❌
        Ok(new_subscriber) => new_subscriber,
        Err(_) => return HttpResponse::InternalServerError, // ?? ❌
    };

    if insert_subscriber(&db_pool, &new_subscriber)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    
    if email_client.send_email(
        &new_subscriber.email, // ?? ❌
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    HttpResponse::Ok().finish()
    
}
```

**Actual Implementation: _Misses_**
```Rust
pub async fn subscribe(
    form: web::Form<FormData>,
    db_pool: web::Data<PgPool>,
    email_client: web::Data<PgPool>,
) {
    let new_subscriber = match form.0.try_into() {
        Ok(form) => form,
        Err(_) => HttpResponse::BadRequest().finish(),
    };

    // [...]
    
    if email_client.send_email(
        new_subscriber.email, // Notice how this call consumes `new_subscriber`
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    // [...]
}
```

#### 7.7.2. A Static Confirmation Link

##### 7.7.2.0. Overview

**Why?** <br/>
We want to ensure that when a user is sent the confirmation email it has a confirmation link

**How?**
- We add a test that checks that the confirmation email that received by a new subscriber it has a confirmation link. For now we can use a dummy link.
- Update the `HtmlBody` and `TextBody` to include a a clickable link.

##### 7.7.2.1. Red Test

**What?**<br/>
- What is the name of the test?
  - _expected_: `subscribe_sends_confirmation_email_with_confirmation_link`
  - _actual_: `subscribe_sends_confirmation_email_with_a_link`
  - _diff_: `confirmation_link`
- What is the test source?<br/>
_Expected_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    // Wire up mock email server
    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    // Trigger confirmation email send
    app.post_subscriptions(body.into()).await;

    // Act
    let email_request = &app.email_server.received_request().unwrap(); // ?? ❌

    // Extract link from email
    let get_link = |s: &str| { // ?? ❌
        let link = linkify::FetchUrl().url()
            .filter()
            .collect();
        assert_eq!(link.len(), 1);
    };

    let html_body = get_links(&email_request["HtmlBody"]).unwrap()[0]; // ?? ❌
    let text_body = get_links(&email_request["TextBody"]).unwrap()[0]; // ?? ❌
    
    // Assert
    assert_eq!(html_body, text_body);
}
```
_Actual_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    // wire up mock email server
    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    // Act
    // trigger email send
    app.post_subscriptions(body.into()).await;
    
    // Assert
    
    // Get the first intercepted request
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    // Parse the body as JSON, starting from raw bytes
    let body: serde_json::Value = serde_json::from_slice(&email_request.body)
        .unwrap();

    // closure to extract link from string
    let get_links = |s: &str| {
        let links: Vec<_> = linkify::LinkFinder::new()
            .links(s)
            .filter(|l| *l.kind() == linkify::LinkKind::Url)
            .collect();
        assert_eq!(links.len(), 1);
        links[0].as_str().to_owned()
    };

    let html_body = get_links(&body["HtmlBody"].as_str().unwrap());
    let text_body = get_links(&body["TextBody"].as_str().unwrap());

    assert_eq!(html_body, text_body);
}
```

_Diff_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // [...]

    // Assert
    // Intercepting first request to email_server
    let email_request = &app.email_server.received_requests().await.unrap()[0];
    // Parse the body as JSON, starting from raw bytes.
    let body: serde_json::Value = serde_json::from_slice(&email_request.body).unwrap();

    let get_links = |s: &str| {
        let links: Vec<_> = linkify::LinkFinder::new()
            .links(s)
            .filter(| l | *l.king() == linkify::LinkKing::Url )
            .collect();
        assert_eq!(links.len(), 1);
        links[0].as_str().to_owned()
    };

    let html_body = get_links(&body["HtmlBody"].as_str().unwrap());
    let text_body = get_links(&body["TextBody"].as_str().unwrap());

    assert_eq!(html_body, text_body);
}
```

##### 7.7.2.2. Green Test

**What?**
- Add a dummy `confirmation_link` to `email_client.send_email()`

_Expected_
```Rust
pub async fn subscribe(/* */) -> HttpResponse {
    //  [...]
    let confirmation_link = "https://dummy-placeholder-domain.com/subscriptions/confirm";
    if email_client
        .send_email(
        new_subscriber.email
        "Welcome!"
        format!(
            "Welcome to our newsletter!<br />\
            Click <a href=\"{}\">here</a> to confirm your subscription.",
            confirmation_link
        ),
        format!(
           "Welcome to our newsletter!\nVisit {} to confirm your subscription.",
            confirmation_link,
        ),
    )
    .await
    .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }
}
```
_Actual_: ✅

_Diff_: 👌

##### 7.7.2.3. Refactor

**What?**<br/>
- extract out `email_client.send_email` into its seprate function that we call in `subscribe`

**Why?**<br/>
- `subscribe` is being polluted by `email_client.send_email` call. We extract it out the call reason about what `subscribe` does more clearly.

#### 7.7.3. Pending Confrimation

##### 7.7.3.0. Overiview

Here, in the book we start by first refactoring the test `subscribe_returns_200_for_valid_form_data` and extracting out the
persistency checks into it own test `subscribe_persists_new_subscriber`. After this we go on to update the new test to assert
that the default `status` for a new subscriber is `pending_confirmation`. 

How we'll do this is a bit different. We will do the refactoring and extracting the persistency checks at the end, following the 
_Red, Green, Refactor_ steps a bit more strictly. This is not necessary. Just doing it this way in an attempt to drill in the TDD
cycle.

_**The Wny?**_<br/>
Here we want to ensure that a new subscriber's `status` is set to `pending_confirmation` to allow us to send a confirmation email
with at confirmation link. When the new subscriber clicks on the confirmaation link that when we update their `status` to confirmed.

_**The How?**_<br/>
Right now the default `status` is `confirmed`. We need to change that.

##### 7.7.3.1. Red Test

_**The What?**_<br/>

We add a test to ensure that the initial `status` of a new subscriber is `pending_confirmation` when they are first inserted into our db.<br/>
For now we will just add an assertion to `subscribe_returns_200_for_valid_form_data` to check the appropriate status for our _Red step._

_Expected_
```Rust
async fn subscribe_returns_200_for_valid_form_data() {
    // [...]
    // We update the query.
    let saved = sqlx.query!("SELECT username, email, status FROM subscriptions",) // ❌
        .fetch_one(&db_pool) // ❌
        .await
        .expect("Failed to execute db query");  // ❌

    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Actual_
```Rust
async fn subscribe_returns_200_for_valid_form_data(){
    // [...]

    // Update our query with status
    let saved = sqlx::query!("SELECT email, username, status FROM subscriptions")
        .fetch_one(&app.db_pool)
        .await
        .expect("Failed to fetch saved subscription in test");

    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Diff_
```Rust
let saved = sqlx::query!("SELECT email, username, status FROM subscriptions", )
    .fetch_one(&app.db_pool)
    .await
    .expect("Failed to fetch saved subscription in test");
```

##### 7.7.3.2. Green Test

_**The What**_<br/>
Here it is very simple. We update the `insert_subscribe` function default from the initial `confirm` to `pending_confirmation`.

_Expected_
```Rust
pub async fn insert_subscriber(db_pool: PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> { // ❌
    // [...]
    sqlx.query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUES ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        Uuid::now_v7(),
        new_subscriber.email, // ❌
        mew_subscriber.username, // ❌
        Utc::now(),
    )
    .execute(&db_pool) // ❌
    .await
    .map_err(|e| {
        // [...]
    })?;

    Ok(())
}
```

_Actual_
```Rust
pub async fn insert_subscriber(db_pool: &PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> {
    sqlx.query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUES ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        Uuid::now_v7(),
        new_subscriber.email.as_ref(),
        mew_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(db_pool)
    .await
    .map_err(|e| {
        // [...]
    })?;

    Ok(())
}
```

_Diff_
```Rust
pub async fn insert_subscriber(db_pool: &PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> {
    sqlx.query!(
        // [...]
        new_subscriber.email.as_ref(),
        new_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(&db_pool)
    .await
    // [...]

    Ok(())
}
```

##### 7.7.3.3. Refactor

_**What?**_<br/>

- Seperate out the persistency assertion from `subscribe_returns_200_for_valid_form_data` into separete `subscribe_persists_new_subscriber`

_**Why?**_<br/>
We want the intention for each test to be clear. On tests the response status the other checks that the data was persisted correctly.

_**How?**_

_Expected_
```Rust
#[tokio::test]
async fn subscribe_returns_200_for_valid_form_data() {
    // [...]
    // Assert
    assert_eq!(response.status().as_u16(), 200);

    // Removes the rest
}

#[tokio::test]
async fn subscribe_persists_new_subscriber() {
    let app = spawn_app().await;
    let body = "usernamelei%20yin&email=lei_yin_loo%40.gmail.com";

    Mock.given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    app.post_subscriptions(body.into()).await;

    let saved = sqlx::query!("SELECT email, username, status FROM subscriptions")
        .fetch_one(&app.db_pool)
        .await
        .except("Failed to fetch saved subscriber in test");

    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Actual_: ✅

_Diff_: 👌

#### 7.7.4. Skeleton of `GET /subscriptions/confirm`

##### 7.7.4.0. Overview

_**The Why?**_<br/>
At the end we want when someone clicks the `confirmation_link` in the confirmation email, a `GET /subscriptions/confirm` request with a
`user_token` is triggered we update the subscriber who clicked from a `pending_confirmation` `status` to a `confirmed` status.

##### 7.7.4.1. Red

_**How?**_
We start with the most simplest version of this by adding a test that checks that when the `confirmation_link` is clicked we get a `200 OK`.

_**What?**_
- What is the name of the test?
    - _Expected - `confirmation_link_click_returns_200_for_valid_token`_
    - _Actual_ - `confirmation_without_token_are rejected_with_400` 
    - _Diff_ - `confirmation_without_token_are_rejected_with_400`

> So here, I actually errored in the whole purpose of this section.
>
> The primary purpose of this section is to start with ensuring a confirmation link has to include a token for us to form a valid request
> to confirm a user. That is why the test we add here checks that if a `confirmation_link` does not include a token it is rejected

_Expected_
```Rust
#[tokio::test]
async fn confirmation_without_token_is_rejected_with_400() {
    // Arrange
    let app = spawn_app().await;

    // Act
    let reponse = reqwest::Client::new()
        .get("{}/subscriptions/confirmation") // ?? ❌
        .send()
        .await
        .expect("Failed to execute request to confirm in test");

    // Assert
    assert_eq!(response.status().as_u16(), 400);
}
```


_Actual_
```Rust
#[tokio::test]
async fn confirmaation_without_tokens_is_rejected_with_400() {
    // Arrange
    let app = spawn_app().await;

    // Act
    let response = reqwest::Client::new()
        .get(format!("{}/subscriptions/confirm", &app.address))
        .send()
        .unwrap();

    // Assert
    assert_eq!(response.status().as_u16(), 400);
}
```

_Diff_
```Rust
async fn confirmations_without_token_are_rejected_with_400(){
    //[...]
    let response = reqwest::Client::new()
        .get(format!("{}/subscriptions/confirm", app.address))
        //[...]
    //[...]
}
```

##### 7.7.4.2. Green

_**What**_<br/>
- We add a `subscriptions_confirm` handler and update it to our routes.
- We ensure the handler expects a `subscription_token` parameter
_**How**_<br/>

_Expected_
```Rust
//! src/routes/mod.rs
// [...]

mod subscriptions_confirm;

pub use subscriptions_confirm::*;


//! src/routes/subscriptions_confirm.rs
use actix_web::Query; // ❌

#[serde(Deserialize)] // ❌
pub struct Params{
    pub subscription_token: String, // ❌
}

// ❌
async fn confirm(_params: Query<Params>) -> HttpResponse { // ❌
    HttpResponse::Ok().finish()
}


//! src/starup.rs

async fn run([...]) -> Result<(), std::io::Error>{ // ❌
    let app = Routes::new() // ❌
            .register("/subscriptions/confirm", get().with(confirm)) // ❌
}
```

_Actual_
```Rust
//! src/routes/mod.rs ✅

//! src/routes/subscriptions_confirm.rs 
use actix_web::HttpResponse;

#[derive(serde::Deserialize)]
pub struct Params {
    subscription_token: String,
}

#[tracing::instrument(
    name = "Confirm pending subscriber",
    skip(_params)
)]
pub async fn confirm(_params: web::Query<Params>) -> HttpResponse {
    HttpResponse::Ok().finish()
}

//! src/startup.rs 
use crate::routes::confirm;

fn run(/**/) -> Result<Server, std::io::Error> {
    //[...]
    let server = HttpServer::new( move || {
        App::new()
            // [...]
            .route("/subscriptions/confirm", web::get().to(confirm))
            // [...]
    })
    // [...]
}
```

_Diff_
```Rust
//! src/routes/subscriptions_confirm.rs

use actix_web::HttpResponse;

#[derive(serde::Deserialize)]
pub struct Params {
    subscription_token: String,
}

#[tracinn::instrument(
    name = "Confirm pending subscriber",
    skip(_params)
)]
pub async fn confirm(_params: web::Query<Params>) -> HttpResponse {
    HttpResponse::Ok().finish()
}

//! src/startup.rs
use crate::routes::confirm;

fn run([...]) -> Result<Server, std::io::Error> {
    // [...]
    let server = HttpServer::new(move || {
        App::new()
            // [...]
            .route("/subscriptions/confirm", web::get().to(confirm))
            // [...]
    })
    // [...]
}
```

#### 7.7.5. Connecting The Dots

##### 7.7.5.0. Overview

This section is a bit bulky. Or is it chunky? In short this section is loooong.

To compress everything and try to get a general idea of what we want to achieve in this sub-section let's try and break down interms of 
firstly...<br/>

_**The Why?**_<br/>
In this section we are building on all the ground work we have done up to now such that when one clicks the `confirmation_link` contained
in the confirmation email that is sent when one becomes a subscriber of our newsletter, we want to perform a `GET` request that returns a
`200 OK`. What makes this section a little bulky is;
1. The test.
    We wire up a mock email server that will make a mock request/send a mock email to a "subscriber". We intercept the reqwest to ensure the
   email has a confirmation link. We then parse the `confirmation_link` to perform a `GET` request ensuring that it returns a 200
2. The core logic
   This feels like the easy part because we just make it so that we pass a `base_url` to our `send_confirmation_email` function with a default toke
   for now.
3. The supporting logic
    We have to make the `base_url` configurable because we are working with a different domain in deployment and we want to setup our implementation in such
   a way that it will work both in production and in local development when testing. This means we need to update our `src/config.rs` `configuration/base.yaml`
   and `src/startup.rs` to ensure that they all accept a configurable `base_url` value.
   We then need to update our `subscribe` method to be able to extract tne `base_url` value from the application context for us to build up our confirmation link
4. Test updating
   We then need to update our `tests/api/helpers.rs` `TestApp` to have a port field that we will use later to set the port to our `confirmation_link` when testing.
   Because in production our DNS manages the port mapping and only have a domain. In testing we need to set the port for us to make a valid request.

##### 7.7.5.1. Red Test

_**What?**_<br/>
What is the name of the test?
- `confirmation_link_returns_200_when_clicked`.

_**How is it implemented?**_<br/>
_Expected_
```Rust
#[tokio::test]
async fn confirmation_link_returns_200_when_clicked() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo@gmail.com";

    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    app.post_subscriptions(body.into());

    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let body: serde_json::Value = serde_json::from_slice(&email_request.body).unwrap();

    let get_link = |s : &str| {
        let links: Vec<_> = linkify::LinkFinder::new()
            .links(s)
            filter( |l| *l.kind() == linkify::LinkKind::Url )
            .collect();
        assert_eq!(links.len() , 1);
        links[0].as_str().to_owned()
    };

    let link = get_link(&body["HtmlBody"].as_str().unwrap()); // ❌
    let confirmation_link = reqwest::Url::parse(link).unwrap();
    
    // Act
    let response = reqwest::Client::new() // ❌
        .get(confirmation_link)
        .send()
        .await
        .expect("Failed to execute confirmation request in test");
    
    // Assert
    assert_eq!(response.status().as_u16(), 200);
}
```

_Actual:_ ~ 94% correct

_Diff_
```Rust
// [...]
let link = &get_link(&body["HtmlBody"].as_str().unwrap());
// [...]
let response = reqwest::get(confirmation_link).await.unwrap();
```

##### 7.7.5.2. Green Test

_**What?**_<br/>
Here we need to update a couple of things. Lets break it down by the files that need updating.
1. `src/routes/subscriptions.rs` - updating `send_confirmation_email` to have configurable base url. We need to update `subscribe` as well
   to pull the `base_url` from the application context.
2. `src/config.rs` - we need to add `base_url` to the `AppSettings` struct
3. `configuration/base.yaml` - we add the `base_url` field
4. `src/startup` - We need to add `ApplicationBaseUrl` new type to wrap the `base_url` and pass it to the application context.
5. `tests/api/helpers` - We need to update the `TestApp` struct to include a `port` field that allows us to construct a valid `confirmation_link`
   with a `port` when running the test.

_**How?**<br/>
We'll sequence the updates in the following order $3, 2, 4, 1, 5$.

3. `configuration/base.yaml` - adding base.url
_Expected_
```YAML
#! configuration/base.yaml
app:
    port: 8000
    base_url: "127.0.0.1" # ❌
# [...]
```

_Actual_
```YAML
#! configuration/base.yaml
app:
    base_url: "http://127.0.0.1"
```

_Diff_
```YAML
#! configuration/base.yaml
app:
    base_url: "http://127.0.0.1"
```

2. `src/config.rs` adding `base_url` field to `AppSettings

_Expected_
```Rust
//! src/config.rs

// [...]
pub struct AppSettings {
    // [...]
    base_url: String,
    
}
```

_Actual_: ✅

_Diff_: ✅


4. `src/startup.rs` - Adding `ApplicationBaseUrl` new type to wrap the `base_url` from config.

_Expected_
```Rust
//! src/startup.rs
// [...]

impl Application {
    async fn build([...]) -> Result<Self, std::io::Error> {
        // [...]

        let server = run(
            // [...]
            base_url // ❌
        )
        .await;// ❌
    }
}

pub struct ApplicationBaseUrl(String); // ❌

async fn run( // ❌
    // [...]
    base_url: String,
) -> Result<Server, std::io::Error> {
    let base_url = web::Data::new(ApplicationBaseUrl(base_url));
    let server = HttpServer::new( move || {
        App::new()
            // [...]
            .app_data(base_url) // ❌
    }); // ❌
    // [...]
}
```

_Actual_
```Rust
//! src/startup.rs
// [...]

impl Application {
    async fn build([...]) -> Result<Self, std::io::Error> {
        // [...]

        let server = run(
            // [...]
            config.app.base_url,
        )?;
        
    }
}

pub struct ApplicationBaseUrl(pub String);

async fn run(
    // [...]
    base_url: String,
) -> Result<Server, std::io::Error> {
    let base_url = web::Data::new(ApplicationBaseUrl(base_url));
    let server = HttpServer::new( move || {
        App::new()
            // [...]
            .app_data(base_url.clone())
    })
    // [...]
}
```

_Diff_
```Rust
//! src/startup.rs
// [...]
    let server = run (
        // [...]
        config.app.base_url
    )?;
// [...]

pub struct ApplicationBaseUrl(pub String);

// [...]
fn run (
    // [...]
    base_url: String
) -> Result<Server, std::io::Error> {
    // [...]
    let server = HttpServer::new(move || {
        App::new()
            //[...]
            .app_data(base_url.clone())
    })
    // [...]
}
```

1. `src/routes/subscriptions` - capture `base_url` from application context.

_Expected_
```Rust
//! src/routes/subscriptions
// [...]


// [...]
pub async fn subscribe(
    // [...]
    base_url: web::Data<ApplicationBaseUrl>
) -> HttpResponse {
    // [...]
    if send_confirmation_email(
        // [...],
        base_url.0 //  ❌
    )
        .await
        .is_err()
    {
        HttpResponse::InternalServerError().finish()
    }

    // [...]
}

//  ❌
async fn send_confirmation_email(
    email_client: &EmailClient, 
    new_subscriber: NewSubscriber,
    base_url: String, //  ❌
) {
    let confirmation_link = format!("{}/subscriptions/confirm", base_url);
    // [...]
}
```

_Actual_
```Rust
//! src/routes/subscriptions
// [...]
pub async fn subscribe(
    // [...]
    base_url: web::Data<ApplicationBaseUrl>
) -> HttpResponse {
    // [...]
    if send_confirmation_email(
        // [...],
        &base_url.0
    )
        .await
        .is_err()
    {
        HttpResponse::InternalServerError().finish()
    }

    // [...]
}

#[tracing::instrument(
    // [...]
    skip(email_client, new_subscriber, base_url)
)]
async fn send_confirmation_email(
    email_client: &EmailClient, 
    new_subscriber: NewSubscriber,
    base_url: &str, 
) {
    let confirmation_link = format!("{}/subscriptions/confirm", base_url);
    // [...]
}
```

_Diff_
```Rust
//! src/routes/subscriptions
// [...]
async fn send_confirmation_email(
    // [...]
    base_url: &str, 
) {
    // [...]
}
```

5. `tests/api/helpers` - Update `TestApp` with `port` field

_Expected_
```Rust
//! tests/api/helpers.rs
// [...]

pub struct TestApp {
    // [...]
    port: u16 //  ❌
}

pub fn spawn_app() -> TestApp {
    // [...]
    let app = Application.build(config).await.expect("Failed to build app in test");//  ❌
    let port = app.port();
    // [...]

    TestApp {
        address,
        db_pool,
        email_client,
        port,
    }
}

//! tests/api/subscriptions_confirm.rs

#[tokio::test]
async fn confirmation_link_returns_200_when_clicked() {
    // [...]
    //  ❌
    assert_eq!(confirmation_link.host_str(), "127.0.0.1");//  ❌
    confirmation_link.set_port(app.port()).unwrap();//  ❌

    // [...]
}
```

_Actual_
```Rust
//! tests/api/helpers.rs

pub struct TestApp {
    // [...]
    pub port: u16,
}

pub fn spawn_app() -> TestApp {
    // [...]
    let app = Application.build(config.clone()).await.expect("Failed to build app in test");
    let port = app.port();
    // [...]

    TestApp {
        address,
        db_pool,
        email_client,
        port,
    }
}

//! tests/api/subscriptions_confirm.rs

#[tokio::test]
async fn confirmation_link_returns_200_when_clicked() {
    // [...]
    let mut confirmation_link = reqwest::Url::parse(raw_confirmation_link).uwnrap();
    assert_eq!(confirmation_link.host_str().unwrap(, "127.0.0.1");
    confirmation_link.set_port(Some(app.port)).unwrap();
    // [...]
}
```

_Diff_
```Rust
//! tests/api/helpers.rs

pub TestApp {
    // [...]
    pub port: u16,
}

pub async fn spawn_app() -> TestApp {
    // [...]
    let app = Application.build(configuration.clone()).awaitl.expect("Failed to build app in test");
    // [...]
}

//! test/api/subscriptions_confirm.rs
#[tokio::test]
async fn confirmation_link_returns_200_if_called() {
    // [...]
    let mut confirmation_link = reqwest::Url::parse(raw_confirmation_link).unwrap();
    assert_eq!(confrimation_link.host_str().unwrap(), "127.0.0.1");
    confirmation_link.set_port(Some(app.port)).unwrap();
    // [...]
}
```

##### 7.7.5.3. Refactor

_**What?**_<br/>
The refactor that we do here mainly extracts out the logic for the `get_link` closure amd makes it part of the `TestApp` methods, making the 
tests a little more cleaner.

__**How?**<br/>
1. We add a `ConfirmationLinks` struct that houses both `html` and `plain_text` links.
2. Add a `get_confirmation_links` method to `TestApp`

_Expected_
```Rust
//! tests/api/helpers.rs

pub struct ConfirmationLinks {
    pub html: reqwest::Url,
    pub plain_text: reqwest::Url,
}

impl TestApp {
    // [...]
    pub async fn get_confirmation_links(body: serde_json::Value) -> ConfirmationLinks { // ❌
        let get_links = |s: &str| { // ❌
            let links: Vec<_> = linkify::LinkFinder::new()
                .links(s)
                .filter(|l| *l.kind() == linkify::LinkKind::Url)
                .collect();
            assert_eq!(links.len(), 1);
            links[0].as_str().to_owned()
             // ❌ 
             // ❌
             // ❌
        };

        let html_link = &get_links(body["HtmlBody"].as_str().unwrap());
        let text_link = &get_links(body["TextBody"].as_str().unwrap());
        ConfirmationLink {
            html: html_link,
            plain_text: text_link
        }
    }
}

//! tests/api/subscriptions.rs
// [...]
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_links() {
    // [...]
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let body: serde_json::Value = serde_json::from_slice(&email_request.body).unrwarp(); //❌ 

    let confirmation_links = app.get_confirmation_links(body); //❌ 

    assert_eq!(confirmation_links.html, confirmation_links.plain_text);
}

//! tests/api/subscriptions.rs
// [...]

#[tokio::test]
async fn confirmation_link_returns_200_when_called() {
    // [...]
    
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let body: serde_json::Value = serde_json::from_slice(&email_request.body).unrwarp(); //❌ 

    
    let confirmation_links = app.get_confirmation_links(body); //❌ 

    let confirmation_link = reqwest::Url::parse(&confirmation_link.html).unwrap(); //❌ 

    let response = reqwest::get(confirmation_link).await.unwrap();

    assert_eq!(response.status().as_u16(), 200);
    
}
```

_Actual_
```Rust
//! tests/api/helpers.rs

pub struct ConfirmationLinks {
    pub html: reqwest::Url,
    pub plain_text: reqwest::Url,
}

impl TestApp {
    // [...]
    pub async fn get_confirmation_link(self, request: &wiremock::Request) -> ConfirmationLink {
        let body: serde_json::Value = serde_json::from_slice(&request.body).unwrap();

        let get_link = |s: &str| {
            let links: Vec<_> = linkify::LinkFinder::new()
                .links(s)
                .filter(|l| *l.kind() == linkify::LinkKind::Url )
                .collect();
            assert_eq!(links.len(), 1);
            let raw_link = links[0].as_str().to_owned();
            let mut confirmation_link = reqwest::Url::parse(&raw_link);
            assert_eq!(confirmation_link.host_str().unwrap(), "127.0.0.1");
            confirmation_link.set_port(Some(self.port)).unwarp();
            confirmation_link
        };

        let html = get_link(&body["HtmlBody"].as_str().unwrap());
        let plain_text = get_link(&body["TextBody"].as_str().unwrap());
        ConfirmationLinks {
            html,
            plain_text,
        }
        
    }
}

//! tests/api/subscriptions.rs
// [...]

#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_links() {
    // [...]
    let email_request = &app.email_server.received_requests().await.unwarp()[0];
    let confirmation_links = app.get_confirmation_links(&email_request);

    // The two links should be identical
    assert_eq!(confirmation_links.html, confirmation_links.plain_text);
}

//! tests/api/subscriptions.rs
// [...]

#[tokio::test]
async fn confirmation_link_returns_200_when_called() {
    // [...]

    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let confirmation_links = app.get_confirmation_links(&email_request);

    // Act
    let response = reqwest::get(confirmation_links.html).await.unwrap();

    // Assert
    assert_eq!(response.status().as_u16(), 200);
}
```

_Diff_
```Rust
//! tests/api/helpers.rs
// [...]

impl TestApp {
    pub async fn get_confirmation_link(&self, request: &wiremock::Request) -> ConfirmationLinks {
        let body: serde_json::Value = serde_json::from_slice(&request.body).unwrap();
        let get_link = |s: &str| {
            // [...]
            let raw_link =  links[0].as_str().to_owned();
            let mut confirmation_link = reqwest::Url::parse(raw_link).unwrap();
            assert_eq!(confirmation_link.host_str().unwarp(), "127.0.0.1");
            confirmation_link.set_port(Some(app.port)).unwrap();
            confirmation_link
        }
    }
}

//! tests/api/subscriptions.rs
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // [...]
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let confirmation_links = app.get_confirmation_links(&email_request);

}

//! tests/api/subscriptions_confirm.rs
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // [...]
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    let confirmation_links = app.get_confirmation_links(&email_request);

}
```

#### 7.7.6. Subscription Tokens

##### 7.7.6.0. Overview

_**Why?**_<br/>
When a user clicks on a `confirmation_link` we want to `status` of a new subscriber to change from `pending_confirmation` to confirm.
To do this the confirmation has to have a unique `subscription_token` that ties a new subscriber to a confirmation link so that when the link is clicked we know
which user because of the unique token.

So the primary objective of this section is to finally generate a token that we will persist to the DB and validate against a confirmation request, allowing us
to appropriately update a user's `status`.

##### 7.7.6.1. Red Test

_**What?**_<br/>
What is the name of the test?
- _Expected:_ - `clicking_confirmation_link_confirms_subscriber`
- _Actual:_ ✅

_**How?**_<br/> 
How is the test implemented.

_Expected_
```Rust
//! tests/api/subscriptions_confirm.rs

#[tokio::test]
async fn clicking_confirmation_link_confirms_new_subscriber() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo@gmail.com";

    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;
    
    app.post_subscriptions(body.into()).await;
    let email_request = &app.email_request.received_requests().await.unwrap()[0];
    let confirmation_links = app.get_confirmation_links(&email_request.body);
    
    // Act
    reqwest::get(confirmation_link)
        .await
        .err_for_status()
        .expect("Failed to execute confirmation request in test");
    
    let updated = sqlx.query!("SELECT username, status FROM subscriptions",)
        .fetch_one(&app.db_pool)
        .await
        .expect("Failed to fetch confirmed user in test");
    
    // Assert
    assert_eq!(updated.status, "confirmed");
    
}
```

- _Actual_: ✅

##### 7.7.6.2. Green Test

_**What?**_<br/>

Here we have a couple of steps that need to be done.
1. Generate a `subscription_token`
2. Pass it as an argument to `send_subscription_email`
3. Add an `store_token` method that inserts an entry into `subscriptions_token` with the new subscriber's `id` and generated `subscription_token`
4. Implement `confirm` in `src/routes/subscriptions_confirm` that is responsible for updating a user from `pending_confirmation` to `confirmed` once handle is called.

_**How?**_<br/>
_Expected_
```Rust
//! src/routes/subscriptions.rs
// [...]
use rand::{distr::Alphanumeric, RngExt};

#[tracing::instrument(
    // [...]
)]
pub async fn subscribe(/**/) -> HttpResponse {
    // [...]
    let user_id = match insert_subscriber(&db_pool, &new_subscriber).await {
        Ok(user_id) => user_id,
        Err(_) => return HttpResponse::InternalServerError().finish();
    };

    let subcription_token = generate_token();

    if store_token(&db_pool, &user_id, &subscription_token) // ❌
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }

    if send_confirmation_email(
        new_subscriber.email, // ❌
        subject, // ❌
        html_body, // ❌
        text_body, // ❌
        &subscription_token
    )
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }

    // [...]
        
}

fn generate_token() -> String {
    std::iter::repeat_with(|| { 
        rand::rng().sample(Alphanumeric)
             // ❌
            .take(25)
            .collect()
    })
}

async fn store_token(
    db_pool: &PgPool
    subscription_token: String,
    user_id: Uuid
) -> Result<(), sqlx::Error> {
    sqlx::query!(
        r#"
            INSERT INTO subscription_tokens (subscription_token, user_id) 
            VALUES ($1, $2)
        "#,
        subscription_token,
        user_id
    )
    .execute(db_pool)
    .await
    .map_err(|e| {
        tracing::error("Error inserting into subscription_tokens: {}", e);
        e
    })?;

    Ok(())
}

async fn insert_subscriber(db_pool: &PgPool, new_subscriber: &NewSubscriber) -> Result<Uuid, sqlx::Error> {
    let subscriber_id = Uuid::now_v7();
    sqlx::query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUES ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        &subscriber_id, // ❌
        new_subscriber.email.as_ref(),
        new_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(db_pool)
    .await
    .map_err(|e| {
        tracing::error("Error inserting new subscriber into subscriptions: {}", e);
        e
    })?;
    Ok(subscriber_id)    
}
```

_Actual_
```Rust
//! src/routes/subscriptions_confirm.rs

#[tracing::instrument([...])]
pub async fn subscribe([...]) -> HttpResponse {
    let sub_id = match insert_subscriber(&db_pool, &new_subscriber).await {
        Ok(subscriber_id) = subscriber_id,
        Err(_) => HttpResponse::InternalServerError().finish()
    };

    let subscription_token = get_subscription_token();

    if store_token(&db_pool, subscriber_id, &subscription_token)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }

    if send_subscription_email(
        &email_server,
        new_subscriber,
        &base_url.0
        &subscription_token
    )
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }
    HttpResponse::Ok().finish()
}

fn generate_subscription_token() -> String {
    std::iter::repeat_with(|| rand::rng().sample(Alphanumeric))
        .map(char::from)
        .take(25)
        .collect()
}

#[tracing::instrument([...])]
async fn insert_subscriber(db_pool: &PgPool, new_subscriber: &NewSubscriber) -> Result<Uuid, sqlx::Error> {
    let subscriber_id = Uuid::now_v7();
    sqlx::query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUE ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        subscriber_id,
        new_subscriber.email.as_ref(),
        new_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(db_pool)
    .await
    .map_err(|e| {
        tracing::error!("Failed to execute insert subscriber query: {:?}", 3);
        e
    })?;
    Ok(subscriber_id)
}

#[tracing::instrument(
    name = "Store subscription_token in the database",
    skip(db_pool, subscription_token, subscriber_id)
)]
async fn store_token(db_pool: &PgPool, subscription_token: &str, subscriber_id: Uuid) -> Result<(), sqlx::Error>{
   sqlx::query!(
       r#"
           INSERT INTO subscription_tokens (subscription_tokens, subscriber_id) 
           VALUES ($1, $2)
       "#,
       subscription_tokens,
       subscriber_id
   )
    .execute(db_pool)
    .await
    .map_err(|e| {
        tracing::error!("Failed to execute store tokens insertion query: {:?}", e);
        e
    })?;
    Ok(())
}

async fn send_subscription_email(
    email_client: &EmailClient,
    new_subscriber: NewSubscriber,
    base_url: &str, 
    subscription_token: &str
) -> Result<(), reqwest::Error> {
    let confirmation_link = format!("{}/subscriptions/confirm?subscription_token={}", base_url, subscription_token);
    // [...]
}
```

_Diff (key diff)_
```Rust
// [...]
pub async fn subscribe([...]) -> HttpResponse {

    // [...]
    if store_token(&db_pool, user_id, &subscription_token)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }

    // [...]
    if send_confirmation_email(
        &email_client,
        new_subscriber,
        &base_url.0,
        &subscription_token
    )
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
}
```

_Expected_<br/>
```Rust
//! src/routes/subscriptions_confirm.rs

#[tracing::instrument([...])]
pub async fn confirm(params: Query<Params>) -> HttpResponse {
    let subscription_token = params.subscription_token;

    let user_id = match get_subscriber_id(subscription_token).await { // ❌
        Some(user_id) => user_id, // ❌
        None => return HttpResponse::InternalServerError().finish(); // ❌
    };

    if update_subscriber_to_confirmed(user_id) // ❌
        .await // ❌
        .is_err()  // ❌
    {
        return HttpResponse::InternalServerError().finish(); // ❌
    }

    HttpResponse::Ok().finish() // ❌
}

// THE REST COULDN'T REMEMBER QUITE WELL. // ❌ 
```

_Actual_ <br/>
```Rust
//! src/routes/subscriptions_confirm.rs

#[tracing::instrument([...])]
pub async fn confirm(
    db_pool: web::Data<PgPool>,
    params: web::Query<Params>,
) -> HttpResponse {
    let id = match get_subscriber_id_from_token(
        &db_pool,
        &params.subscription_token,
    ).await {
        Ok(id) => id,
        Err(_) => return HttpResponse::InternalServerError().finish(),
    };

    match id {
        // Non-existing token!
        None => HttpResponse::Unauthorized().finish(),
        Some(subscriber_id) => {
            if confirm_subscriber(&db_pool, subscriber_id).await.is_err() {
                return HttpResponse::InternalServerError().finish();
            }
            HttpResponse::Ok().finish()
        }
    }
}

#[tracing::instrument(
    name = "Get subscriber_id from token"
    skip(db_pool, subscriber_token)
)]
async fn get_subsriber_id(db_pool: &PgPool, subscription_token: &str) -> Result<Option<Uuid>, sqlx::Error> {
    let result = sqlx::query!(
        r#"
            SELECT subscriber_id 
                FROM subscription_tokens
            WHERE subscription_token = $1
        "#,
        subscription_token
    )
    .fetch_optional(db_pool)
    .await
    .map_err(|e| {
        tracing::error!("Failed to execute get_subscriber_id query: {:?}", e)
        e
    })?;

    Ok(result.map(|r| r.subscriber_id))
}

#[tracing::instrument(
    name="Mark subscriber as confirmed"
    skip(db_pool, subscriber_id)
)]
async fn confirm_subscriber(db_pool: &PgPool, subscriber_id: Uuid) -> Result<(), sqlx::Error> {
    sqlx::query!(
        r#"
            UPDATE subscriptions
                SET status = 'confirmed'
            WHERE id = $1
        "#,
        subscriber_id
    )
    .execute(db_pool)
    .await
    .map(|e| {
        tracing::error!("Failed to execute confirm_subscriber query: {:?}", e);
        e
    })?;
    Ok(())
}
```

_Diff_: **Nearly everything 🙆‍♂️**

###### Misc notes

It is important note that the `rand` crate has been updated a bit since the book came out. 

The book uses `rand = "0.8"` we are now at`rand = "0.10.1"`. So some API names are different.

To read more the necessary migration when using a higher version of rand than that in the book take a look at the extended
`rand` documentation [here](https://rust-random.github.io/book/update-0.9.html).

Below we go through a small snippet

In [12]:
:dep rand

In [13]:
{
    use rand::distr::Alphanumeric;
    use rand::RngExt;

    let res: String = std::iter::repeat_with(|| rand::rng().sample(Alphanumeric))
        .map(char::from)
        .take(25)
        .collect();
    println!("res: {res:?}");
};

res: "KJX3J8UAjMyXA5YM3m3Qu6ILd"


### 7.7.x Axum

## 7.8. Database Transactions

### 7.7.x. Actix

#### 7.8.0. Overview

_**Why?**_<br/>
The primary goal here is to ensure that `insert_subscriber` and `store_token` happen in a single transaction because for
every `new_subscriber` we want a `subscription_token` that we use to confirm a subscription. Both operations should succeed
and therefore both operations need to be atomic

#### 7.8.1. Refactor

_**What?**_<br/>
That means we have to do a refactor of the `subscribe` handler to create a mutable `transaction` that will be passed to both the
`insert_subscriber` and `store_token` functions. Then commit the `transaction` if both operations were successful.

_**How?**_<br/>

_Expected_

```Rust
//! src/routes/subscriptions.rs

// [...]
use sqlx::Transaction; // ❌

// [...]
pub async fn subscribe([...]) -> HttpResponse {
    // [...]
    let transaction = match db_pool.begin().await {
        Ok(transaction) => transaction,
        Err(_) => HttpResponse::InternalServerError().finish(),
    };

    let subscriber_id = match insert_subscriber( &mut transaction, &new_subscriber ).await {
        Ok(subscriber_id) => subscriber_id,
        Err(_) => HttpResponse::InternalServerError().finish(),
    };

    // [...]

    if store_token(&mut transaction, &subscription_token, subscriber_id)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    };

    // [...]

    if transaction.commit().await.is_err() {
        return HttpResponse::InternalServerError().finish();
    };

    // [...]

}

#[tracing::instrument(
    name = "Adding a new subscriber to Database",
    skip(transaction, new_subscriber),
)]
async fn insert_subscriber(
    transaction: &mut Transaction<'_, Query>, // ❌
    new_subscriber: &NewSubscriber,
) -> Result<Uuid, sqlx::Error> {
    let subscriber_id = Uuid::now_v7();
    let query = sqlx::query!([...]);
    transaction.execute(query)
        .await
        .map_err(|e| {
            // [...]
        })?;
    Ok(subscriber_id)
}

#[tracing::instrument(
    name = "Store subscription_token in Database",
    skip(transaction, subscription_tokne, subscriber_id),
)]
async fn store_token(
    transaction: &mut Transaction<'_, Query> // ❌
    subscription_token: &str,
    subscriber_id: Uuid,
) -> Result<(), sqlx::Error> {
    let query = sqlx::query!([...]);
    transaction.execute(query)
        .await
        .map_error(|e| {
            // [...]
        })?;
    Ok(())
}
```

_Actual:_ ~ 95% correct

_Diff_: 

```Rust
//! src/routes/subscriptions.rs
use sqlx::{Postgres, Transactions, Executor};

// [...]

#[tracing::instrument([...])]
async fn insert_subscriber(
    transaction: &mut Transaction<'_, Postgres>,
    new_subscriber: &new_subscriber,
) -> Result<Uuid, sqlx::Error>  {
    // [...]
    Ok(subscriber_id)
}


#[tracing::instrument([...])]
async fn store_token(
    transaction: &mut Transaction<'_, Postgres>,
    subscription_token: &str,
    subscriber_id: Uuid,
) -> Result<(), sqlx::Error> {
    // [...]
}
```